# 🏆 SOTA Bank Telemarketing Prediction
## Target: Top 3 (0.948+)

This notebook implements a state-of-the-art ensemble approach:
- Multiple CatBoost configurations with diverse features
- LightGBM for model diversity
- Target encoding with CV-based approach
- Advanced feature engineering
- Optimized rank-based blending

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

print('Libraries loaded successfully!')

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

TARGET = 'Subscribed'
ID_COL = 'id'
FOLDS = 5
SEEDS = [42, 2026, 1337, 777, 314]  # 5 seeds for stability

BASE_CAT = [
    'job', 'marital', 'education', 'default', 'housing',
    'loan', 'contact', 'month', 'poutcome',
]

MONTH_ORDER = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12,
}

# CatBoost configs
CB_PARAMS = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'iterations': 2500,
    'learning_rate': 0.035,
    'depth': 6,
    'l2_leaf_reg': 5.0,
    'random_strength': 0.8,
    'bootstrap_type': 'Bayesian',
    'bagging_temperature': 0.15,
    'od_type': 'Iter',
    'od_wait': 200,
    'allow_writing_files': False,
    'verbose': 100,
    'thread_count': -1,
}

CB_BERN_PARAMS = {
    **CB_PARAMS,
    'bootstrap_type': 'Bernoulli',
    'subsample': 0.8,
}

CB_DEEP_PARAMS = {
    **CB_PARAMS,
    'depth': 8,
    'learning_rate': 0.025,
    'l2_leaf_reg': 3.0,
}

# LightGBM config
LGBM_PARAMS = {
    'objective': 'binary',
    'metric': 'auc',
    'n_estimators': 2500,
    'learning_rate': 0.035,
    'num_leaves': 31,
    'max_depth': 6,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 5.0,
    'colsample_bytree': 0.8,
    'subsample': 0.85,
    'subsample_freq': 1,
    'verbose': -1,
    'n_jobs': -1,
    'force_col_wise': True,
}

In [ ]:
# ============================================================================
# UTILITY FUNCTIONS
# ============================================================================

def locate_file(filename):
    candidates = [
        Path('/kaggle/input/fiicode-2026-ai-competition') / filename,
        Path(filename),
        Path.cwd() / filename,
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(f'Could not find {filename}')

def rank_norm(values):
    return pd.Series(values).rank(method='average', pct=True).to_numpy()

def finalize_cats(df, cat_cols):
    for col in cat_cols:
        df[col] = df[col].fillna('missing').astype(str)
    return df

In [ ]:
# ============================================================================
# FEATURE ENGINEERING V1 - Minimal Features
# ============================================================================

def build_features_v1(df):
    """Minimal but effective feature set"""
    df = df.copy()
    if ID_COL in df.columns:
        df = df.drop(columns=[ID_COL])
    
    for col in BASE_CAT:
        df[col] = df[col].astype(str)
    
    # Basic transforms
    df['month_num'] = df['month'].str.lower().map(MONTH_ORDER).fillna(0).astype(np.int16)
    df['has_previous'] = (df['pdays'] != -1).astype(np.int8)
    df['pdays_clean'] = np.where(df['pdays'] == -1, 999, df['pdays']).astype(np.int16)
    
    # Log transforms
    df['balance_log'] = np.sign(df['balance']) * np.log1p(np.abs(df['balance']))
    df['duration_log'] = np.log1p(df['duration'].clip(lower=0))
    df['campaign_log'] = np.log1p(df['campaign'].clip(lower=0))
    df['previous_log'] = np.log1p(df['previous'].clip(lower=0))
    
    # Interactions
    df['job_education'] = df['job'].astype(str) + '__' + df['education'].astype(str)
    df['contact_month'] = df['contact'].astype(str) + '__' + df['month'].astype(str)
    
    df['age_band'] = pd.cut(
        df['age'], bins=[0, 25, 35, 45, 55, 65, 120],
        labels=['<=25', '26-35', '36-45', '46-55', '56-65', '65+'],
    ).astype(str)
    
    cat_cols = BASE_CAT + ['job_education', 'contact_month', 'age_band']
    return finalize_cats(df, cat_cols), cat_cols

In [ ]:
# ============================================================================
# FEATURE ENGINEERING V2 - Advanced Features
# ============================================================================

def build_features_v2(df):
    """Advanced feature engineering with domain knowledge"""
    df = df.copy()
    if ID_COL in df.columns:
        df = df.drop(columns=[ID_COL])
    
    for col in BASE_CAT:
        df[col] = df[col].astype(str)
    
    month = df['month'].str.lower()
    job = df['job'].astype(str)
    marital = df['marital'].astype(str)
    education = df['education'].astype(str)
    contact = df['contact'].astype(str)
    poutcome = df['poutcome'].astype(str)
    loan = df['loan'].astype(str)
    default = df['default'].astype(str)
    housing = df['housing'].astype(str)
    
    df['month'] = month
    df['month_num'] = month.map(MONTH_ORDER).fillna(0).astype(np.int16)
    
    # Seasonal features
    df['is_q4'] = month.isin(['oct', 'nov', 'dec']).astype(np.int8)
    df['is_summer'] = month.isin(['jun', 'jul', 'aug']).astype(np.int8)
    
    # pdays processing
    df['pdays_was_missing'] = (df['pdays'] == -1).astype(np.int8)
    df['pdays_clean'] = df['pdays'].replace(-1, 999).astype(np.int16)
    
    # Log transforms
    df['duration_log1p'] = np.log1p(df['duration'].clip(lower=0))
    df['balance_abs_log1p'] = np.log1p(df['balance'].abs())
    df['balance_signed_log1p'] = np.sign(df['balance']) * np.log1p(df['balance'].abs())
    df['campaign_log1p'] = np.log1p(df['campaign'].clip(lower=0))
    df['previous_log1p'] = np.log1p(df['previous'].clip(lower=0))
    df['pdays_log1p'] = np.log1p(df['pdays_clean'])
    
    # Interaction features - numeric
    df['contacts_total'] = df['campaign'] + df['previous']
    df['duration_per_campaign'] = df['duration'] / (df['campaign'] + 1)
    df['balance_per_age'] = df['balance'] / (df['age'] + 1)
    df['previous_per_campaign'] = df['previous'] / (df['campaign'] + 1)
    df['campaign_x_previous'] = df['campaign'] * df['previous']
    df['duration_x_campaign'] = df['duration'] * df['campaign']
    df['duration_per_contact'] = df['duration'] / (df['contacts_total'] + 1)
    
    # Binary features
    df['has_any_loan'] = ((housing == 'yes') | (loan == 'yes')).astype(np.int8)
    df['has_both_loans'] = ((housing == 'yes') & (loan == 'yes')).astype(np.int8)
    df['is_default'] = (default == 'yes').astype(np.int8)
    df['was_contacted_before'] = (df['pdays'] != -1).astype(np.int8)
    df['is_cellular'] = (contact == 'cellular').astype(np.int8)
    df['balance_negative'] = (df['balance'] < 0).astype(np.int8)
    df['prev_success'] = (poutcome == 'success').astype(np.int8)
    df['prev_failure'] = (poutcome == 'failure').astype(np.int8)
    
    # Bucketed features
    df['age_bucket'] = pd.cut(
        df['age'], bins=[0, 25, 35, 45, 55, 65, 120],
        labels=['<=25', '26-35', '36-45', '46-55', '56-65', '65+'],
    ).astype(str)
    
    df['campaign_bucket'] = pd.cut(
        df['campaign'], bins=[-1, 1, 2, 4, 9, np.inf],
        labels=['1', '2', '3-4', '5-9', '10+'],
    ).astype(str)
    
    df['previous_bucket'] = pd.cut(
        df['previous'], bins=[-1, 0, 1, 3, np.inf],
        labels=['0', '1', '2-3', '4+'],
    ).astype(str)
    
    pdays_source = df['pdays'].replace(-1, 999)
    df['pdays_bucket'] = pd.cut(
        pdays_source, bins=[-1, 7, 30, 90, 365, np.inf],
        labels=['<=1w', '8-30d', '31-90d', '91-365d', '365d+'],
    ).astype(str)
    df.loc[df['pdays'] == -1, 'pdays_bucket'] = 'never'
    
    df['duration_bucket'] = pd.cut(
        df['duration'], bins=[-1, 60, 120, 240, 480, 900, np.inf],
        labels=['<=1m', '1-2m', '2-4m', '4-8m', '8-15m', '15m+'],
    ).astype(str)
    
    df['balance_bucket'] = pd.cut(
        df['balance'], bins=[-np.inf, -500, 0, 500, 1500, 5000, np.inf],
        labels=['<-500', '-500-0', '0-500', '500-1500', '1500-5000', '5000+'],
    ).astype(str)
    
    df['day_bucket'] = pd.cut(
        df['day'], bins=[0, 10, 20, 31],
        labels=['early', 'mid', 'late'], include_lowest=True,
    ).astype(str)
    
    # Categorical interactions
    df['job_education'] = job + '__' + education
    df['job_marital'] = job + '__' + marital
    df['contact_month'] = contact + '__' + month
    df['poutcome_month'] = poutcome + '__' + month
    df['loan_default'] = loan + '__' + default
    df['loan_housing'] = loan + '__' + housing
    df['housing_default'] = housing + '__' + default
    df['contact_day_bucket'] = contact + '__' + df['day_bucket'].astype(str)
    df['month_day_bucket'] = month + '__' + df['day_bucket'].astype(str)
    df['history_state'] = np.where(df['previous'] > 0, poutcome + '__seen', 'no_previous')
    df['age_bucket_contact'] = df['age_bucket'].astype(str) + '__' + contact
    df['age_bucket_history'] = df['age_bucket'].astype(str) + '__' + df['history_state'].astype(str)
    df['contact_history_state'] = contact + '__' + df['history_state'].astype(str)
    df['month_pdays_bucket'] = month + '__' + df['pdays_bucket'].astype(str)
    df['job_contact'] = job + '__' + contact
    df['education_contact'] = education + '__' + contact
    df['job_age_bucket'] = job + '__' + df['age_bucket'].astype(str)
    df['duration_contact'] = df['duration_bucket'].astype(str) + '__' + contact
    df['balance_housing'] = df['balance_bucket'].astype(str) + '__' + housing
    
    cat_cols = BASE_CAT + [
        'age_bucket', 'campaign_bucket', 'previous_bucket', 'pdays_bucket',
        'duration_bucket', 'balance_bucket', 'day_bucket',
        'job_education', 'job_marital', 'contact_month', 'poutcome_month',
        'loan_default', 'loan_housing', 'housing_default',
        'contact_day_bucket', 'month_day_bucket', 'history_state',
        'age_bucket_contact', 'age_bucket_history', 'contact_history_state',
        'month_pdays_bucket', 'job_contact', 'education_contact',
        'job_age_bucket', 'duration_contact', 'balance_housing',
    ]
    
    return finalize_cats(df, cat_cols), cat_cols

In [ ]:
# ============================================================================
# TARGET ENCODING
# ============================================================================

def add_target_encoding(train_df, test_df, y, cat_cols, smoothing=10.0):
    """CV-based target encoding to avoid leakage"""
    train_df = train_df.copy()
    test_df = test_df.copy()
    global_mean = y.mean()
    
    for col in cat_cols:
        if col not in train_df.columns:
            continue
        
        te_col = f'{col}_te'
        train_df[te_col] = np.nan
        
        kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        for train_idx, valid_idx in kf.split(train_df, y):
            fold_train = train_df.iloc[train_idx]
            fold_y = y.iloc[train_idx]
            
            stats = fold_train.groupby(col).apply(
                lambda x: (fold_y.loc[x.index].sum() + smoothing * global_mean) / 
                          (len(x) + smoothing)
            )
            
            train_df.iloc[valid_idx, train_df.columns.get_loc(te_col)] = (
                train_df.iloc[valid_idx][col].map(stats).fillna(global_mean)
            )
        
        stats = train_df.groupby(col).apply(
            lambda x: (y.loc[x.index].sum() + smoothing * global_mean) / 
                      (len(x) + smoothing)
        )
        test_df[te_col] = test_df[col].map(stats).fillna(global_mean)
    
    return train_df, test_df


def add_frequency_encoding(train_df, test_df, cat_cols):
    """Add frequency encoding for categorical features"""
    train_df = train_df.copy()
    test_df = test_df.copy()
    
    combined = pd.concat([train_df, test_df], axis=0, ignore_index=True)
    n_total = len(combined)
    
    for col in cat_cols:
        if col not in train_df.columns:
            continue
        
        counts = combined[col].value_counts()
        freq = counts / n_total
        
        train_df[f'{col}_freq'] = train_df[col].map(freq).fillna(0).astype(np.float32)
        train_df[f'{col}_count'] = train_df[col].map(counts).fillna(0).astype(np.int32)
        test_df[f'{col}_freq'] = test_df[col].map(freq).fillna(0).astype(np.float32)
        test_df[f'{col}_count'] = test_df[col].map(counts).fillna(0).astype(np.int32)
    
    return train_df, test_df

In [ ]:
# ============================================================================
# MODEL TRAINING FUNCTIONS
# ============================================================================

def train_catboost(x_train, x_test, y, cat_cols, params, seeds, use_class_weight=False, name='CB'):
    """Train CatBoost with multiple seeds"""
    oof = np.zeros(len(x_train), dtype=float)
    test_pred = np.zeros(len(x_test), dtype=float)
    
    for seed in seeds:
        seed_oof = np.zeros(len(x_train), dtype=float)
        seed_test = np.zeros(len(x_test), dtype=float)
        folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
        
        for fold_idx, (train_idx, valid_idx) in enumerate(folds.split(x_train, y), 1):
            train_x = x_train.iloc[train_idx].reset_index(drop=True)
            train_y = y.iloc[train_idx].reset_index(drop=True)
            valid_x = x_train.iloc[valid_idx].reset_index(drop=True)
            valid_y = y.iloc[valid_idx].reset_index(drop=True)
            
            fold_params = dict(params)
            fold_params['verbose'] = False
            if use_class_weight:
                fold_params['auto_class_weights'] = 'Balanced'
            
            model = CatBoostClassifier(**fold_params, random_seed=seed)
            model.fit(
                train_x, train_y,
                eval_set=(valid_x, valid_y),
                cat_features=cat_cols,
                use_best_model=True,
                verbose=False,
            )
            
            seed_oof[valid_idx] = model.predict_proba(valid_x)[:, 1]
            seed_test += model.predict_proba(x_test)[:, 1] / FOLDS
            
            fold_auc = roc_auc_score(valid_y, seed_oof[valid_idx])
            print(f'    {name} seed {seed} fold {fold_idx} AUC: {fold_auc:.6f}')
        
        seed_auc = roc_auc_score(y, seed_oof)
        print(f'  {name} seed {seed} full AUC: {seed_auc:.6f}')
        oof += seed_oof / len(seeds)
        test_pred += seed_test / len(seeds)
    
    return oof, test_pred


def train_lightgbm(x_train, x_test, y, cat_cols, params, seeds, name='LGBM'):
    """Train LightGBM with multiple seeds"""
    oof = np.zeros(len(x_train), dtype=float)
    test_pred = np.zeros(len(x_test), dtype=float)
    
    x_train_lgb = x_train.copy()
    x_test_lgb = x_test.copy()
    for col in cat_cols:
        if col in x_train_lgb.columns:
            x_train_lgb[col] = x_train_lgb[col].astype('category')
            x_test_lgb[col] = x_test_lgb[col].astype('category')
    
    for seed in seeds:
        seed_oof = np.zeros(len(x_train), dtype=float)
        seed_test = np.zeros(len(x_test), dtype=float)
        folds = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=seed)
        
        for fold_idx, (train_idx, valid_idx) in enumerate(folds.split(x_train, y), 1):
            train_x = x_train_lgb.iloc[train_idx].reset_index(drop=True)
            train_y = y.iloc[train_idx].reset_index(drop=True)
            valid_x = x_train_lgb.iloc[valid_idx].reset_index(drop=True)
            valid_y = y.iloc[valid_idx].reset_index(drop=True)
            
            fold_params = dict(params)
            fold_params['random_state'] = seed
            
            model = LGBMClassifier(**fold_params)
            model.fit(train_x, train_y, eval_set=[(valid_x, valid_y)])
            
            seed_oof[valid_idx] = model.predict_proba(valid_x)[:, 1]
            seed_test += model.predict_proba(x_test_lgb)[:, 1] / FOLDS
            
            fold_auc = roc_auc_score(valid_y, seed_oof[valid_idx])
            print(f'    {name} seed {seed} fold {fold_idx} AUC: {fold_auc:.6f}')
        
        seed_auc = roc_auc_score(y, seed_oof)
        print(f'  {name} seed {seed} full AUC: {seed_auc:.6f}')
        oof += seed_oof / len(seeds)
        test_pred += seed_test / len(seeds)
    
    return oof, test_pred

In [ ]:
# ============================================================================
# BLENDING FUNCTIONS
# ============================================================================

def grid_search_blend(predictions, y, step=100):
    """Grid search for optimal blend weights"""
    names = list(predictions.keys())
    n = len(names)
    
    best_auc = -1.0
    best_weights = {}
    
    if n == 2:
        for a in range(step + 1):
            w = [a / step, (step - a) / step]
            blend = sum(w[i] * rank_norm(predictions[names[i]]) for i in range(n))
            auc = roc_auc_score(y, blend)
            if auc > best_auc:
                best_auc = auc
                best_weights = {names[i]: w[i] for i in range(n)}
    
    elif n == 3:
        for a in range(step + 1):
            for b in range(step + 1 - a):
                c = step - a - b
                w = [a / step, b / step, c / step]
                blend = sum(w[i] * rank_norm(predictions[names[i]]) for i in range(n))
                auc = roc_auc_score(y, blend)
                if auc > best_auc:
                    best_auc = auc
                    best_weights = {names[i]: w[i] for i in range(n)}
    
    elif n >= 4:
        # Coarser grid for more models
        step = 20
        for a in range(0, step + 1, 2):
            for b in range(0, step + 1 - a, 2):
                for c in range(0, step + 1 - a - b, 2):
                    remaining = step - a - b - c
                    if n == 4:
                        w = [a / step, b / step, c / step, remaining / step]
                    else:
                        for d in range(0, remaining + 1, 2):
                            e = remaining - d
                            w = [a / step, b / step, c / step, d / step, e / step]
                            blend = sum(w[i] * rank_norm(predictions[names[i]]) for i in range(n))
                            auc = roc_auc_score(y, blend)
                            if auc > best_auc:
                                best_auc = auc
                                best_weights = {names[i]: w[i] for i in range(n)}
                        continue
                    blend = sum(w[i] * rank_norm(predictions[names[i]]) for i in range(n))
                    auc = roc_auc_score(y, blend)
                    if auc > best_auc:
                        best_auc = auc
                        best_weights = {names[i]: w[i] for i in range(n)}
    
    return best_auc, best_weights


def apply_blend(predictions, weights):
    result = np.zeros(len(next(iter(predictions.values()))), dtype=np.float64)
    for name, preds in predictions.items():
        result += weights.get(name, 0.0) * rank_norm(preds)
    return result

In [ ]:
# ============================================================================
# LOAD DATA
# ============================================================================

train = pd.read_csv(locate_file('train.csv'))
test = pd.read_csv(locate_file('test.csv'))
y = train[TARGET].astype(int)
train_features = train.drop(columns=[TARGET])

print(f'Train shape: {train.shape}')
print(f'Test shape: {test.shape}')
print(f'Target distribution: {y.mean():.4f}')

In [ ]:
# ============================================================================
# TRAIN MODELS
# ============================================================================

oof_results = {}
test_results = {}

# Model 1: CatBoost V1 (minimal features)
print('\n' + '='*60)
print('[1] CatBoost V1 - Minimal Features')
print('='*60)
x_train_v1, cat_cols_v1 = build_features_v1(train_features)
x_test_v1, _ = build_features_v1(test)

cb_v1_oof, cb_v1_test = train_catboost(
    x_train_v1, x_test_v1, y, cat_cols_v1,
    CB_PARAMS, SEEDS, name='CB_V1',
)
auc = roc_auc_score(y, cb_v1_oof)
print(f'\n>>> CB_V1 Final AUC: {auc:.6f}')
oof_results['cb_v1'] = cb_v1_oof
test_results['cb_v1'] = cb_v1_test

In [ ]:
# Model 2: CatBoost V2 (advanced features + class weights)
print('\n' + '='*60)
print('[2] CatBoost V2 - Advanced Features + Class Weights')
print('='*60)
x_train_v2, cat_cols_v2 = build_features_v2(train_features)
x_test_v2, _ = build_features_v2(test)

cb_v2_oof, cb_v2_test = train_catboost(
    x_train_v2, x_test_v2, y, cat_cols_v2,
    CB_PARAMS, SEEDS, use_class_weight=True, name='CB_V2',
)
auc = roc_auc_score(y, cb_v2_oof)
print(f'\n>>> CB_V2 Final AUC: {auc:.6f}')
oof_results['cb_v2'] = cb_v2_oof
test_results['cb_v2'] = cb_v2_test

In [ ]:
# Model 3: CatBoost Bernoulli
print('\n' + '='*60)
print('[3] CatBoost Bernoulli')
print('='*60)

cb_bern_oof, cb_bern_test = train_catboost(
    x_train_v2, x_test_v2, y, cat_cols_v2,
    CB_BERN_PARAMS, SEEDS, use_class_weight=True, name='CB_BERN',
)
auc = roc_auc_score(y, cb_bern_oof)
print(f'\n>>> CB_BERN Final AUC: {auc:.6f}')
oof_results['cb_bern'] = cb_bern_oof
test_results['cb_bern'] = cb_bern_test

In [ ]:
# Model 4: CatBoost Deep
print('\n' + '='*60)
print('[4] CatBoost Deep')
print('='*60)

cb_deep_oof, cb_deep_test = train_catboost(
    x_train_v2, x_test_v2, y, cat_cols_v2,
    CB_DEEP_PARAMS, SEEDS[:3], use_class_weight=True, name='CB_DEEP',
)
auc = roc_auc_score(y, cb_deep_oof)
print(f'\n>>> CB_DEEP Final AUC: {auc:.6f}')
oof_results['cb_deep'] = cb_deep_oof
test_results['cb_deep'] = cb_deep_test

In [ ]:
# Model 5: LightGBM with target encoding
print('\n' + '='*60)
print('[5] LightGBM with Target Encoding')
print('='*60)

te_cols = ['job', 'education', 'month', 'poutcome', 'contact', 'job_education', 'contact_month']
x_train_v2_te, x_test_v2_te = add_target_encoding(x_train_v2.copy(), x_test_v2.copy(), y, te_cols)
freq_cols = ['job', 'education', 'month', 'contact', 'poutcome']
x_train_v2_te, x_test_v2_te = add_frequency_encoding(x_train_v2_te, x_test_v2_te, freq_cols)

lgbm_oof, lgbm_test = train_lightgbm(
    x_train_v2_te, x_test_v2_te, y, cat_cols_v2,
    LGBM_PARAMS, SEEDS, name='LGBM',
)
auc = roc_auc_score(y, lgbm_oof)
print(f'\n>>> LGBM Final AUC: {auc:.6f}')
oof_results['lgbm'] = lgbm_oof
test_results['lgbm'] = lgbm_test

In [ ]:
# ============================================================================
# ENSEMBLE & BLEND
# ============================================================================

print('\n' + '='*60)
print('ENSEMBLE BLENDING')
print('='*60)

# Find optimal blend weights
best_auc, best_weights = grid_search_blend(oof_results, y, step=50)
print(f'\nOptimal blend weights:')
for name, w in sorted(best_weights.items(), key=lambda x: -x[1]):
    print(f'  {name}: {w:.3f}')
print(f'\nOptimal blend AUC: {best_auc:.6f}')

# Apply blend
blend_oof = apply_blend(oof_results, best_weights)
blend_test = apply_blend(test_results, best_weights)

# Equal weight blend as backup
equal_weights = {k: 1/len(oof_results) for k in oof_results}
equal_oof = apply_blend(oof_results, equal_weights)
equal_auc = roc_auc_score(y, equal_oof)
print(f'Equal weight blend AUC: {equal_auc:.6f}')

In [ ]:
# ============================================================================
# GENERATE SUBMISSIONS
# ============================================================================

print('\n' + '='*60)
print('GENERATING SUBMISSIONS')
print('='*60)

# Final submission
submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: np.clip(blend_test, 1e-6, 1.0 - 1e-6),
})
submission.to_csv('submission.csv', index=False)
print('Saved: submission.csv')

# Individual model submissions for analysis
for name, test_pred in test_results.items():
    sub = pd.DataFrame({
        ID_COL: test[ID_COL],
        TARGET: np.clip(rank_norm(test_pred), 1e-6, 1.0 - 1e-6),
    })
    sub.to_csv(f'submission_{name}.csv', index=False)
    print(f'Saved: submission_{name}.csv')

print('\n' + '='*60)
print(f'🎯 FINAL CV AUC: {best_auc:.6f}')
print('='*60)

submission.head(10)